# Lab 2 : End to end — chat with your own docs

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.


## What we are achieving in this lab

**Objective.** Lab 1 stopped at search. The only new step is **generate**: put the retrieved chunks in a prompt and ask Claude to answer **only** from them.

**Prerequisites.** Lab 1 finished. `OPENAI_API_KEY` (embeddings) and `ANTHROPIC_API_KEY` (chat). Both in the `.env` file in this folder.

**The line so far.**

| Lab | What you already did |
|-----|----------------------|
| 1 | Chunk, embed, cosine, keyword vs meaning, store, `retriever.invoke` |
| 2 (this lab) | Take those hits. Generate an answer from them. |

**What this lab uses.**

| Layer | What it does | What we use |
|-------|----------------|-------------|
| Split | Cut the handbook | `RecursiveCharacterTextSplitter` (500 / 50) — Lab 1 |
| Embed | Chunk → vector | OpenAI `text-embedding-3-small` — Lab 1 |
| Store | Keep vectors | `InMemoryVectorStore` — Lab 1 |
| Retrieve | Closest chunks | `retriever.invoke` — Lab 1 |
| Generate | Answer from those chunks | Claude `claude-haiku-4-5` — **new** |

Claude does not embed. OpenAI does not write this answer.

**What you will do.**

1. Index `handbook.txt` the Lab 1 way. Print how many chunks were stored.
2. Retrieve for the reimbursement question. **Print the chunks** before you generate.
3. Send chunks + question to Claude. Read the answer against the chunks.
4. Ask something the handbook does not contain. The retriever still returns hits. The prompt should refuse.

**Cost.** A few embeddings plus two short Claude calls. Fractions of a cent.


## Where this sits after Lab 1

Lab 1 stopped at search. RAG's last letter is **generation**.

```
load → split → embed → store → retrieve → generate
                 Lab 1                      this lab
```

Index (split, embed, store) runs when the handbook changes.
Question time is: retrieve, then generate.

No LangGraph today. LangGraph is a way to wire retrieve-then-generate later. The loop does not change.


### Step 1. Both keys

OpenAI for embeddings. Anthropic for the chat model. If either cell fails, fix `.env` from Lab 1 / Week 2 and restart the kernel.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY") or not os.getenv("ANTHROPIC_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY and ANTHROPIC_API_KEY in the .env file in this folder.")

print("keys : set")


### Step 2. Index (Labs 4 and 5)

Load, split at 500, store. Print the chunk count. This is not a new algorithm.


In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open("handbook.txt", encoding="utf-8") as f:
    HANDBOOK = f.read()

# Lab 1: 500 was the usable size for this file.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunk_texts = splitter.split_text(HANDBOOK)

docs = []
for i, text in enumerate(chunk_texts):
    docs.append(Document(page_content=text, metadata={"chunk": i, "source": "handbook.txt"}))

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# Lab 1: index once.
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("file   : handbook.txt")
print("chunks :", len(docs))
print("embed  :", EMBED_MODEL)
print("store  : InMemoryVectorStore  (this kernel only)")


### Step 3. Retrieve first (Lab 1)

Same question as Labs 4–5. Print the chunks. If these are wrong, the answer will be wrong. A fluent model will hide that.


In [ ]:
QUESTION = "How do I get reimbursed for a $300 train ticket?"

hits = retriever.invoke(QUESTION)

print("Question:", QUESTION)
print()
print("Retrieved chunks (read these before the answer)")
print()
for i, hit in enumerate(hits, start=1):
    print("--- hit", i, "  chunk", hit.metadata.get("chunk"), " ---")
    print(hit.page_content)
    print()


You should see the reimbursement section: finance portal, PDF receipt, travel under $500. A $300 ticket needs that last sentence.

### Step 4. Generate from those chunks only

This is the new step. Claude sees **only** the three hits, not the rest of the handbook.

The system prompt is Week 2: answer from the context. Do not guess.


In [ ]:
from langchain_anthropic import ChatAnthropic

CHAT_MODEL = "claude-haiku-4-5"
llm = ChatAnthropic(model=CHAT_MODEL, temperature=0)

parts = []
for hit in hits:
    parts.append(hit.page_content)
context = "\n\n---\n\n".join(parts)

messages = [
    (
        "system",
        "You answer from a company handbook excerpt. "
        "Use only facts that appear in the context. "
        "If the context does not contain the answer, say you cannot find it. "
        "Be brief.",
    ),
    (
        "human",
        "Context:\n" + context + "\n\nQuestion: " + QUESTION,
    ),
]

answer = llm.invoke(messages)
print("chat model:", CHAT_MODEL)
print()
print("--- grounded answer ---")
print(answer.content)


Grade in this order:

1. Did retrieval return the reimbursement chunk? (Labs 4–5)
2. Did the answer stay inside those chunks? (this lab)

The model never saw parental leave or PTO. That is what **grounded** means: the answer is tied to text you can show someone.


### Step 5. A question the handbook does not answer

The retriever still returns three chunks. It always will (`k=3`). Print them. Then generate.


In [ ]:
missing = "What is the company's pet-bereavement policy?"
missing_hits = retriever.invoke(missing)

print("Question:", missing)
print()
print("Retriever still returned:")
print()
for i, hit in enumerate(missing_hits, start=1):
    print("--- hit", i, "  chunk", hit.metadata.get("chunk"), " ---")
    print(hit.page_content)
    print()

parts = []
for hit in missing_hits:
    parts.append(hit.page_content)
missing_context = "\n\n---\n\n".join(parts)

refusal = llm.invoke(
    [
        (
            "system",
            "You answer using ONLY the provided context. "
            "If the answer is not in the context, say you cannot find it. Do not guess.",
        ),
        (
            "human",
            "Context:\n" + missing_context + "\n\nQuestion: " + missing,
        ),
    ]
)
print("--- model reply ---")
print(refusal.content)


Those hits are the closest *wrong* chunks (often PTO or time off). A grounded prompt should refuse. If a model ever invents a pet policy from a PTO chunk, that is the failure Lab 3 names.

Production later adds a cosine floor so you skip generate when nothing is close enough. Not today.


### Optional. Same loop on a PDF

Drop a short PDF next to this notebook as `handbook.pdf`. `pypdf` reads the text layer. Then the same split → store → retrieve.

Scanned PDFs with no text will print empty pages. That is a parsing problem, not an embeddings problem.


In [ ]:
from pypdf import PdfReader

pdf_path = Path("handbook.pdf")
if not pdf_path.exists():
    pdf_path = Path("week03") / "handbook.pdf"

if not pdf_path.exists():
    print("No handbook.pdf found. Skip this cell, or drop a PDF next to the notebook.")
else:
    pages = []
    reader = PdfReader(str(pdf_path))
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append(
            Document(
                page_content=text,
                metadata={"source": str(pdf_path), "page": i + 1},
            )
        )
    pdf_chunks = splitter.split_documents(pages)
    pdf_store = InMemoryVectorStore.from_documents(pdf_chunks, embedding=embeddings)
    pdf_hits = pdf_store.as_retriever(search_kwargs={"k": 3}).invoke(QUESTION)
    print("pages :", len(reader.pages))
    print("chunks:", len(pdf_chunks))
    print("file  :", pdf_path)
    print()
    for hit in pdf_hits:
        preview = hit.page_content.replace("\n", " ")
        if len(preview) > 140:
            preview = preview[:140] + "..."
        print("p." + str(hit.metadata.get("page")), preview)


## What you should be able to explain

> "RAG is load → split → embed → store → retrieve → generate. I can point at each step in this notebook."

> "I print retrieved chunks before I trust the answer. Wrong chunks, wrong answer."

> "Claude writes the answer. OpenAI embeds. Anthropic has no embedding model."

> "The retriever always returns k chunks. The prompt must say: if it is not in the context, do not guess."

**Try it as a product.** [`handbook-chat/`](./handbook-chat/) is this loop in a Streamlit UI. Run it with Docker.

**Lab 3** keeps this loop and points it at the cases that make a demo look finished when it is not.
